## Install dependencies

This notebook collects additional task settings for the OpenAI model using the Batch API. The outputs are saved in a separate task-set folder so that previously collected data are not overwritten.

In [ ]:
!pip install -U openai pandas tqdm python-dotenv

## Cell 1 — Imports, OpenAI client, and experiment configuration

This cell sets up the OpenAI client, model parameters, run identifier, folder structure, and data-collection constants.

The task-set identifier is included in the output path and filenames so this collection can be combined with earlier data later without overwriting existing files.

In [1]:
from __future__ import annotations

import os
import re
import json
import time
import uuid
import hashlib
import datetime as dt
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# -----------------------------
# OpenAI client
# -----------------------------
# assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY. Put it in your environment or .env file."
client = OpenAI(api_key="")

# -----------------------------
# Experiment configuration
# -----------------------------
PROVIDER = "openai"
MODEL_NAME = "gpt-5.4"
TEMPERATURE = 1.0

# This notebook only collects the new task settings.
# Keep this ID stable so later analysis can identify this task set.
TASK_SET_ID = "taskset_b_additional_prompts"

# Cost/control options for GPT-5.4-style Responses API calls.
REASONING_EFFORT = "none"
TEXT_VERBOSITY = "medium"
PROMPT_CACHE_RETENTION = "24h"
PROMPT_CACHE_KEY = None

N_BASE_AGENTS = 150
N_DYADS = 75
N_TRIADS = 50

MAX_OUTPUT_TOKENS_BY_FAMILY = {
    "slogan": 60,
    "aut": 120,
    "story": 700,
}

# A fresh run_id prevents accidental overwrites.
RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]

DATA_ROOT = (
    Path("ai_data")
    / "deflect_creativity"
    / PROVIDER
    / f"model_{MODEL_NAME}"
    / TASK_SET_ID
    / f"run_{RUN_ID}"
)

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "round1_plans": DATA_ROOT / "01_round1" / "plans",
    "round1_batch_inputs": DATA_ROOT / "01_round1" / "batch_inputs",
    "round1_manifests": DATA_ROOT / "01_round1" / "manifests",
    "round1_raw_outputs": DATA_ROOT / "01_round1" / "raw_outputs",
    "round1_raw_errors": DATA_ROOT / "01_round1" / "raw_errors",
    "round1_parsed": DATA_ROOT / "01_round1" / "parsed",
    "round2_plans": DATA_ROOT / "02_round2" / "plans",
    "round2_batch_inputs": DATA_ROOT / "02_round2" / "batch_inputs",
    "round2_manifests": DATA_ROOT / "02_round2" / "manifests",
    "round2_raw_outputs": DATA_ROOT / "02_round2" / "raw_outputs",
    "round2_raw_errors": DATA_ROOT / "02_round2" / "raw_errors",
    "round2_parsed": DATA_ROOT / "02_round2" / "parsed",
    "compiled": DATA_ROOT / "03_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run directory:")
print(DATA_ROOT)

Run directory:
ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3


## Cell 2 — Utility functions

This cell defines reusable helpers for timestamps, JSON/JSONL I/O, safe filenames, hashes, and model-output cleaning.

In [2]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def safe_slug(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip()
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()
    return text

## Cell 3 — Save run configuration

This cell saves the main configuration values for reproducibility. The saved configuration includes the model, task set, sampling parameters, number of agents, and output folder.

In [3]:
run_config = {
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "task_set_id": TASK_SET_ID,
    "temperature": TEMPERATURE,
    "reasoning_effort": REASONING_EFFORT,
    "text_verbosity": TEXT_VERBOSITY,
    "prompt_cache_retention": PROMPT_CACHE_RETENTION,
    "prompt_cache_key": PROMPT_CACHE_KEY,
    "n_base_agents": N_BASE_AGENTS,
    "n_dyads": N_DYADS,
    "n_triads": N_TRIADS,
    "max_output_tokens_by_family": MAX_OUTPUT_TOKENS_BY_FAMILY,
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "created_at_utc": now_iso(),
}

config_path = DIRS["metadata"] / f"experiment_config__{TASK_SET_ID}__{RUN_ID}.json"
write_json(config_path, run_config)

config_path

PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/00_metadata/experiment_config__taskset_b_additional_prompts__20260518_091520__368abfc3.json')

## Cell 4 — Define new task settings

This cell defines the additional task settings to be collected.

The new settings include one slogan task, three AUT tasks, and two story tasks. The AUT common uses are kept exactly as specified for standardization.

In [4]:
TASK_SETTINGS = [
    {
        "task_id": "slogan_blood_donation",
        "task_family": "slogan",
        "task_label": "Blood donation slogan",
        "task_prompt_key": "blood_donation",
    },
    {
        "task_id": "aut_key",
        "task_family": "aut",
        "task_label": "AUT key",
        "task_prompt_key": "key",
        "object": "key",
        "common_use": "used to open a lock",
    },
    {
        "task_id": "aut_wooden_pencil",
        "task_family": "aut",
        "task_label": "AUT wooden pencil",
        "task_prompt_key": "wooden_pencil",
        "object": "wooden pencil",
        "common_use": "used for writing",
    },
    {
        "task_id": "aut_automobile_tire",
        "task_family": "aut",
        "task_label": "AUT automobile tire",
        "task_prompt_key": "automobile_tire",
        "object": "automobile tire",
        "common_use": "used on the wheel of an automobile",
    },
    {
        "task_id": "story_horror",
        "task_family": "story",
        "task_label": "Horror story",
        "task_prompt_key": "horror",
    },
    {
        "task_id": "story_life_last_seconds",
        "task_family": "story",
        "task_label": "Life and last seconds story",
        "task_prompt_key": "life_last_seconds",
    },
]

STRATEGIES = ["vanilla", "diverge"]
CONDITIONS = ["base", "dyad", "triad"]

tasks_df = pd.DataFrame(TASK_SETTINGS)
tasks_df

,task_id,task_family,task_label,task_prompt_key,object,common_use
0,slogan_blood_donation,slogan,Blood donation slogan,blood_donation,NaN,NaN
1,aut_key,aut,AUT key,key,key,used to open a lock
2,aut_wooden_pencil,aut,AUT wooden pencil,wooden_pencil,wooden pencil,used for writing
3,aut_automobile_tire,aut,AUT automobile tire,automobile_tire,automobile tire,used on the wheel of an automobile
4,story_horror,story,Horror story,horror,NaN,NaN
5,story_life_last_seconds,story,Life and last seconds story,life_last_seconds,NaN,NaN


## Cell 5 — Prompt builders

This cell defines the shared system instruction, the vanilla and diverge strategy blocks, and the task-specific prompts.

The Round 2 prompt structure is parallel across tasks. The base condition shows only the agent's own previous response; the dyad condition adds one peer response; the triad condition adds two peer responses.

In [5]:
SYSTEM_INSTRUCTIONS = (
    "You are participating in a controlled creativity experiment. "
    "Follow the task instructions exactly. Return exactly one response. "
    "Do not explain your reasoning. Do not include commentary before or after the response."
)


def strategy_block(strategy: str) -> str:
    if strategy == "vanilla":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task."
        )

    if strategy == "diverge":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task.\n"
            "- Try to make it stand out from other responses that might be generated for this same task."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def base_task_prompt(task: dict) -> str:
    task_id = task["task_id"]

    if task_id == "slogan_blood_donation":
        return (
            "You are part of the communications team at a nonprofit organization preparing a campaign "
            "to encourage blood donation.\n\n"
            "Generate exactly one campaign slogan for this blood donation campaign.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the campaign.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id in {"aut_key", "aut_wooden_pencil", "aut_automobile_tire"}:
        return (
            "You are participating in a creativity task.\n\n"
            f"Object: {task['object']}\n"
            f"Common use to avoid: {task['common_use']}\n\n"
            "Generate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\n"
            "Requirements:\n"
            "- Do not use the common use.\n"
            "- Do not list multiple uses.\n"
            "- The response must be written in English.\n"
            "- Return only the alternative use as a short phrase or one sentence."
        )

    if task_id == "story_horror":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one short horror story designed to chill the bones.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_life_last_seconds":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story in 8 sentences. The first sentence must describe 100 years of a character's life. "
            "The next 7 sentences must describe the last 10 seconds of that character's life.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not number or label the sentences.\n"
            "- Do not state which sentence does what.\n"
            "- Return only the story as one paragraph."
        )

    raise ValueError(f"Unknown task_id: {task_id}")


def build_round1_prompt(task: dict, strategy: str) -> str:
    return base_task_prompt(task) + "\n\n" + strategy_block(strategy)


def build_round2_context(
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    if condition == "base":
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
        )
    elif condition == "dyad":
        assert len(peer_round1_texts) == 1
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous response from another agent in the same first round:\n'
            f'"{peer_round1_texts[0]}"\n\n'
        )
    elif condition == "triad":
        assert len(peer_round1_texts) == 2
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous responses from two other agents in the same first round:\n'
            f'1. "{peer_round1_texts[0]}"\n'
            f'2. "{peer_round1_texts[1]}"\n\n'
        )
    else:
        raise ValueError(f"Unknown condition: {condition}")

    if strategy == "vanilla":
        return context + "Now generate one new response for the same task."

    if strategy == "diverge":
        return (
            context
            + "Now generate one new response for the same task. "
              "It should stand out from the previous response(s) shown above while still satisfying all task requirements."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def build_round2_prompt(
    task: dict,
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + build_round2_context(
            strategy=strategy,
            condition=condition,
            self_round1=self_round1,
            peer_round1_texts=peer_round1_texts,
        )
    )

## Cell 6 — Build the agent roster

This cell creates the experimental units for the base, dyad, and triad conditions.

Each condition contributes 150 agents: 150 singleton agents in base, 75 dyads in dyad, and 50 triads in triad.

In [6]:
def build_agent_roster() -> pd.DataFrame:
    rows = []

    for i in range(1, N_BASE_AGENTS + 1):
        rows.append({
            "condition": "base",
            "group_id": f"base_{i:03d}",
            "group_size": 1,
            "agent_index": 1,
            "agent_id": f"base_{i:03d}__a1",
        })

    for g in range(1, N_DYADS + 1):
        for a in [1, 2]:
            rows.append({
                "condition": "dyad",
                "group_id": f"dyad_{g:03d}",
                "group_size": 2,
                "agent_index": a,
                "agent_id": f"dyad_{g:03d}__a{a}",
            })

    for g in range(1, N_TRIADS + 1):
        for a in [1, 2, 3]:
            rows.append({
                "condition": "triad",
                "group_id": f"triad_{g:03d}",
                "group_size": 3,
                "agent_index": a,
                "agent_id": f"triad_{g:03d}__a{a}",
            })

    return pd.DataFrame(rows)


agents_df = build_agent_roster()

display(agents_df.groupby("condition").agg(
    n_agents=("agent_id", "count"),
    n_groups=("group_id", "nunique"),
    group_size=("group_size", "first"),
))

agents_df.head()

,n_agents,n_groups,group_size
condition,,,
base,150,150,1
dyad,150,75,2
triad,150,50,3


,condition,group_id,group_size,agent_index,agent_id
0,base,base_001,1,1,base_001__a1
1,base,base_002,1,1,base_002__a1
2,base,base_003,1,1,base_003__a1
3,base,base_004,1,1,base_004__a1
4,base,base_005,1,1,base_005__a1


## Cell 7 — Build the Round 1 request plan

This cell creates the complete Round 1 request plan for all new task settings.

Round 1 contains independent ideation calls: no previous responses or peer responses are shown. The expected number of Round 1 requests is 6 task settings × 2 strategies × 450 agents = 5,400 requests.

In [7]:
def build_round1_plan() -> pd.DataFrame:
    rows = []

    for task in TASK_SETTINGS:
        for strategy in STRATEGIES:
            for _, agent in agents_df.iterrows():
                user_prompt = build_round1_prompt(task, strategy)
                request_basis = {
                    "provider": PROVIDER,
                    "model": MODEL_NAME,
                    "task_set_id": TASK_SET_ID,
                    "round": 1,
                    "task_id": task["task_id"],
                    "task_family": task["task_family"],
                    "strategy": strategy,
                    "condition": agent["condition"],
                    "group_id": agent["group_id"],
                    "agent_id": agent["agent_id"],
                    "agent_index": int(agent["agent_index"]),
                }
                request_key = "r1__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

                rows.append({
                    **request_basis,
                    "request_key": request_key,
                    "system_instructions": SYSTEM_INSTRUCTIONS,
                    "user_prompt": user_prompt,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                    "created_at_utc": now_iso(),
                })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head()}")

    return plan_df


round1_plan_df = build_round1_plan()

expected_round1 = len(TASK_SETTINGS) * len(STRATEGIES) * len(agents_df)

print("Expected Round 1 requests:", expected_round1)
print("Actual Round 1 requests:  ", len(round1_plan_df))

display(round1_plan_df.groupby(["task_id", "strategy", "condition"]).size().reset_index(name="n"))
round1_plan_df.head()

Expected Round 1 requests: 5400
Actual Round 1 requests:   5400


,task_id,strategy,condition,n
0,aut_automobile_tire,diverge,base,150
1,aut_automobile_tire,diverge,dyad,150
2,aut_automobile_tire,diverge,triad,150
3,aut_automobile_tire,vanilla,base,150
4,aut_automobile_tire,vanilla,dyad,150
5,aut_automobile_tire,vanilla,triad,150
6,aut_key,diverge,base,150
7,aut_key,diverge,dyad,150
8,aut_key,diverge,triad,150
9,aut_key,vanilla,base,150


,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,request_key,system_instructions,user_prompt,temperature,max_output_tokens,created_at_utc
0,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_001,base_001__a1,1,r1__f980c6c7e3a4fcf2a108aed6,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:21:24.731456+00:00
1,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_002,base_002__a1,1,r1__c26c483bdc78cf9ab3171f3f,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:21:24.731616+00:00
2,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_003,base_003__a1,1,r1__c23b5ca1af5c87dc557b7974,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:21:24.731756+00:00
3,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_004,base_004__a1,1,r1__05b8b847be5fbdcb2bcdfcee,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:21:24.731889+00:00
4,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_005,base_005__a1,1,r1__503029f248f45d353956e03b,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:21:24.732020+00:00


## Cell 8 — Write the OpenAI Batch JSONL file

This cell writes the Round 1 request plan to a local JSONL file formatted for the OpenAI Batch API with the Responses endpoint.

The local plan and JSONL files are saved before submission for reproducibility.

In [8]:
def make_openai_responses_batch_jsonl(
    plan_df: pd.DataFrame,
    round_name: str,
    output_dir: Path,
) -> tuple[Path, Path]:
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{timestamp}"

    plan_path = output_dir.parent / "plans" / f"{stem}__plan.csv"
    jsonl_path = output_dir / f"{stem}__batch_input.jsonl"

    plan_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    if plan_path.exists() or jsonl_path.exists():
        raise FileExistsError("Refusing to overwrite an existing plan or batch-input file.")

    plan_df.to_csv(plan_path, index=False)

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            body = {
                "model": MODEL_NAME,
                "instructions": row["system_instructions"],
                "input": row["user_prompt"],
                "temperature": float(row["temperature"]),
                "max_output_tokens": int(row["max_output_tokens"]),
            }

            if REASONING_EFFORT is not None:
                body["reasoning"] = {"effort": REASONING_EFFORT}

            if TEXT_VERBOSITY is not None:
                body["text"] = {"verbosity": TEXT_VERBOSITY}

            if PROMPT_CACHE_RETENTION is not None:
                body["prompt_cache_retention"] = PROMPT_CACHE_RETENTION

            if PROMPT_CACHE_KEY is not None:
                body["prompt_cache_key"] = PROMPT_CACHE_KEY

            request = {
                "custom_id": row["request_key"],
                "method": "POST",
                "url": "/v1/responses",
                "body": body,
            }
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"Wrote plan:  {plan_path}")
    print(f"Wrote batch: {jsonl_path}")
    print(f"Requests:    {len(plan_df):,}")

    return jsonl_path, plan_path


round1_jsonl_path, round1_plan_path = make_openai_responses_batch_jsonl(
    plan_df=round1_plan_df,
    round_name="round1",
    output_dir=DIRS["round1_batch_inputs"],
)

round1_jsonl_path, round1_plan_path

Wrote plan:  ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/plans/taskset_b_additional_prompts__round1__openai__gpt-5.4__20260518_092155__plan.csv
Wrote batch: ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/batch_inputs/taskset_b_additional_prompts__round1__openai__gpt-5.4__20260518_092155__batch_input.jsonl
Requests:    5,400


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/batch_inputs/taskset_b_additional_prompts__round1__openai__gpt-5.4__20260518_092155__batch_input.jsonl'),
 PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/plans/taskset_b_additional_prompts__round1__openai__gpt-5.4__20260518_092155__plan.csv'))

## Cell 9 — Submit Round 1 batch

This cell uploads the Round 1 JSONL file and submits it as an OpenAI Batch job.

The batch manifest is saved locally and includes the OpenAI batch ID, uploaded file ID, local input file path, and plan path.

In [9]:
def submit_openai_batch(
    batch_jsonl_path: Path,
    round_name: str,
    plan_path: Path,
    manifest_dir: Path,
) -> dict:
    batch_input_file = client.files.create(
        file=open(batch_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "project": "deflect_creativity",
            "task_set_id": TASK_SET_ID,
            "round": round_name,
            "provider": PROVIDER,
            "model": MODEL_NAME,
            "run_id": RUN_ID,
            "local_input_file": str(batch_jsonl_path),
            "local_plan_file": str(plan_path),
        },
    )

    batch_info = {
        "run_id": RUN_ID,
        "task_set_id": TASK_SET_ID,
        "round": round_name,
        "provider": PROVIDER,
        "model": MODEL_NAME,
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "status_at_submission": batch.status,
        "submitted_at_utc": now_iso(),
        "batch_jsonl_path": str(batch_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = manifest_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__batch_manifest__{batch.id}.json"
    if manifest_path.exists():
        raise FileExistsError(f"Refusing to overwrite manifest: {manifest_path}")

    write_json(manifest_path, batch_info)
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


round1_batch_info = submit_openai_batch(
    batch_jsonl_path=round1_jsonl_path,
    round_name="round1",
    plan_path=round1_plan_path,
    manifest_dir=DIRS["round1_manifests"],
)

round1_batch_info

Submitted batch:
{
  "run_id": "20260518_091520__368abfc3",
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round1",
  "provider": "openai",
  "model": "gpt-5.4",
  "batch_id": "batch_6a0b128f4a3c8190a292c71ae59c032a",
  "input_file_id": "file-KWCfffHq5ae9U1NZmzYM3W",
  "status_at_submission": "validating",
  "submitted_at_utc": "2026-05-18T13:22:24.494681+00:00",
  "batch_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/batch_inputs/taskset_b_additional_prompts__round1__openai__gpt-5.4__20260518_092155__batch_input.jsonl",
  "plan_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/plans/taskset_b_additional_prompts__round1__openai__gpt-5.4__20260518_092155__plan.csv",
  "data_root": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3",
  "manifest_path": "ai_data/

{'run_id': '20260518_091520__368abfc3',
 'task_set_id': 'taskset_b_additional_prompts',
 'round': 'round1',
 'provider': 'openai',
 'model': 'gpt-5.4',
 'batch_id': 'batch_6a0b128f4a3c8190a292c71ae59c032a',
 'input_file_id': 'file-KWCfffHq5ae9U1NZmzYM3W',
 'status_at_submission': 'validating',
 'submitted_at_utc': '2026-05-18T13:22:24.494681+00:00',
 'batch_jsonl_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/batch_inputs/taskset_b_additional_prompts__round1__openai__gpt-5.4__20260518_092155__batch_input.jsonl',
 'plan_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/plans/taskset_b_additional_prompts__round1__openai__gpt-5.4__20260518_092155__plan.csv',
 'data_root': 'ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3',
 'manifest_path': 'ai_data/deflect_creativity/openai/model_

## Cell 10 — Check Round 1 batch status

This cell retrieves the current status of the submitted Round 1 batch.

Run this cell periodically until the batch status is `completed`.

In [13]:
def check_openai_batch(batch_id: str) -> dict:
    batch = client.batches.retrieve(batch_id)

    info = {
        "batch_id": batch.id,
        "status": batch.status,
        "created_at": batch.created_at,
        "in_progress_at": getattr(batch, "in_progress_at", None),
        "finalizing_at": getattr(batch, "finalizing_at", None),
        "completed_at": getattr(batch, "completed_at", None),
        "failed_at": getattr(batch, "failed_at", None),
        "expired_at": getattr(batch, "expired_at", None),
        "cancelled_at": getattr(batch, "cancelled_at", None),
        "request_counts": None,
        "output_file_id": batch.output_file_id,
        "error_file_id": batch.error_file_id,
    }

    if batch.request_counts:
        info["request_counts"] = {
            "total": batch.request_counts.total,
            "completed": batch.request_counts.completed,
            "failed": batch.request_counts.failed,
        }

    usage = getattr(batch, "usage", None)
    if usage:
        try:
            info["usage"] = usage.model_dump()
        except Exception:
            info["usage"] = str(usage)

    print(json.dumps(info, indent=2))
    return info


round1_status = check_openai_batch(round1_batch_info["batch_id"])
round1_status

{
  "batch_id": "batch_6a0b128f4a3c8190a292c71ae59c032a",
  "status": "completed",
  "created_at": 1779110543,
  "in_progress_at": 1779110607,
  "finalizing_at": 1779111454,
  "completed_at": 1779112251,
  "failed_at": null,
  "expired_at": null,
  "cancelled_at": null,
  "request_counts": {
    "total": 5400,
    "completed": 5400,
    "failed": 0
  },
  "output_file_id": "file-GaLPSx4iDZjGuZZMJzucfK",
  "error_file_id": null,
  "usage": {
    "input_tokens": 860400,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 558397,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 1418797
  }
}


{'batch_id': 'batch_6a0b128f4a3c8190a292c71ae59c032a',
 'status': 'completed',
 'created_at': 1779110543,
 'in_progress_at': 1779110607,
 'finalizing_at': 1779111454,
 'completed_at': 1779112251,
 'failed_at': None,
 'expired_at': None,
 'cancelled_at': None,
 'request_counts': {'total': 5400, 'completed': 5400, 'failed': 0},
 'output_file_id': 'file-GaLPSx4iDZjGuZZMJzucfK',
 'error_file_id': None,
 'usage': {'input_tokens': 860400,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 558397,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 1418797}}

## Cell 11 — Optional reload after kernel restart

Use this cell if the notebook kernel is restarted before the Round 1 batch finishes.

Paste the manifest path printed during submission, then run the cell to restore the batch information.

In [ ]:
# Optional reload after kernel restart:
# manifest_path = Path("ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_.../01_round1/manifests/...")
# round1_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(round1_batch_info["data_root"])
# round1_batch_info

## Cell 12 — Download Round 1 raw results

Once the Round 1 batch is completed, this cell downloads the successful output file and the error file if one exists.

The raw JSONL output is saved before parsing.

In [14]:
def download_openai_batch_results(
    batch_id: str,
    raw_output_dir: Path,
    raw_error_dir: Path,
    round_name: str,
) -> tuple[Optional[Path], Optional[Path]]:
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"Batch is not completed yet. Current status: {batch.status}")
        return None, None

    output_path = raw_output_dir / f"{TASK_SET_ID}__{round_name}__{batch_id}__output.jsonl"
    error_path = raw_error_dir / f"{TASK_SET_ID}__{round_name}__{batch_id}__errors.jsonl"

    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite output file: {output_path}")

    if batch.output_file_id:
        file_response = client.files.content(batch.output_file_id)
        output_path.write_text(file_response.text, encoding="utf-8")
        print(f"Downloaded output: {output_path}")

    if batch.error_file_id:
        if error_path.exists():
            raise FileExistsError(f"Refusing to overwrite error file: {error_path}")
        error_response = client.files.content(batch.error_file_id)
        error_path.write_text(error_response.text, encoding="utf-8")
        print(f"Downloaded errors: {error_path}")
    else:
        error_path = None
        print("No error file.")

    return output_path, error_path


round1_output_path, round1_error_path = download_openai_batch_results(
    batch_id=round1_batch_info["batch_id"],
    raw_output_dir=DIRS["round1_raw_outputs"],
    raw_error_dir=DIRS["round1_raw_errors"],
    round_name="round1",
)

round1_output_path, round1_error_path

Downloaded output: ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/raw_outputs/taskset_b_additional_prompts__round1__batch_6a0b128f4a3c8190a292c71ae59c032a__output.jsonl
No error file.


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/raw_outputs/taskset_b_additional_prompts__round1__batch_6a0b128f4a3c8190a292c71ae59c032a__output.jsonl'),
 None)

## Cell 13 — Parse Round 1 results

This cell parses the raw OpenAI Batch output into standardized records and saves parsed files in JSONL, CSV, and pickle formats.

The parsed data preserve the original plan metadata and add model text, status, usage, and batch metadata.

In [15]:
def extract_text_from_responses_api_body(body: dict) -> str:
    if not isinstance(body, dict):
        return ""

    if body.get("output_text"):
        return str(body["output_text"]).strip()

    texts = []
    for item in body.get("output", []) or []:
        for content in item.get("content", []) or []:
            if isinstance(content, dict) and content.get("type") in {"output_text", "text"} and "text" in content:
                texts.append(content["text"])

    return "\n".join(texts).strip()


def parse_openai_batch_output_to_standard_files(
    batch_output_path: Path,
    plan_path: Path,
    parsed_dir: Path,
    round_name: str,
    batch_id: str,
) -> dict:
    plan_df = pd.read_csv(plan_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    batch_records = read_jsonl(batch_output_path)

    parsed_jsonl_path = parsed_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.pkl"

    if parsed_jsonl_path.exists() or parsed_csv_path.exists() or parsed_pkl_path.exists():
        raise FileExistsError("Refusing to overwrite existing parsed files.")

    parsed_records = []
    n_success = 0
    n_empty = 0
    n_error = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})
        response = rec.get("response") or {}
        error = rec.get("error")

        if response and response.get("body"):
            body = response["body"]
            text = clean_model_text(extract_text_from_responses_api_body(body))
            usage = body.get("usage")

            status = "success" if text else "empty_text"
            n_success += int(status == "success")
            n_empty += int(status == "empty_text")

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": status,
                "text": text,
                "provider_response_id": body.get("id"),
                "usage": usage,
                "error": None if text else "No text extracted from response body.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
            }
        else:
            n_error += 1
            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                "error": error,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
            }

        parsed_records.append(record)
        append_jsonl(parsed_jsonl_path, record)

    parsed_df = pd.DataFrame(parsed_records)
    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "task_set_id": TASK_SET_ID,
        "round": round_name,
        "batch_id": batch_id,
        "n_records": len(parsed_df),
        "n_success": n_success,
        "n_empty_text": n_empty,
        "n_error": n_error,
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parse_summary.json"
    write_json(summary_path, summary)

    print(json.dumps(summary, indent=2))
    return summary


round1_parse_summary = parse_openai_batch_output_to_standard_files(
    batch_output_path=round1_output_path,
    plan_path=Path(round1_batch_info["plan_path"]),
    parsed_dir=DIRS["round1_parsed"],
    round_name="round1",
    batch_id=round1_batch_info["batch_id"],
)

round1_df = pd.read_pickle(round1_parse_summary["parsed_pkl_path"])
print(round1_df.shape)
display(round1_df["status"].value_counts(dropna=False))
round1_df.head()

{
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round1",
  "batch_id": "batch_6a0b128f4a3c8190a292c71ae59c032a",
  "n_records": 5400,
  "n_success": 5400,
  "n_empty_text": 0,
  "n_error": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/parsed/taskset_b_additional_prompts__round1__openai__gpt-5.4__batch_6a0b128f4a3c8190a292c71ae59c032a__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/parsed/taskset_b_additional_prompts__round1__openai__gpt-5.4__batch_6a0b128f4a3c8190a292c71ae59c032a__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/01_round1/parsed/taskset_b_additional_prompts__round1__openai__gpt-5.4__batch_6a0b128f4a3c8190a292c71ae59c032a__parsed.pkl"
}
(5400, 26)


status
success    5400
Name: count, dtype: int64

,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,created_at_utc,parsed_at_utc,status,text,provider_response_id,usage,error,batch_custom_id,batch_output_file,batch_id
0,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_001,base_001__a1,...,2026-05-18T13:21:24.731456+00:00,2026-05-18T13:52:05.488707+00:00,success,"Pass Life Forward, One Pint",resp_071bf71c97d56709006a0b13187a048197a463a01...,"{'input_tokens': 136, 'input_tokens_details': ...",None,r1__f980c6c7e3a4fcf2a108aed6,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b128f4a3c8190a292c71ae59c032a
1,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_002,base_002__a1,...,2026-05-18T13:21:24.731616+00:00,2026-05-18T13:52:05.489038+00:00,success,"Be Someone’s Type, Save Life",resp_0ee8a157c97edaf3006a0b13183724819799a7f4b...,"{'input_tokens': 136, 'input_tokens_details': ...",None,r1__c26c483bdc78cf9ab3171f3f,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b128f4a3c8190a292c71ae59c032a
2,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_003,base_003__a1,...,2026-05-18T13:21:24.731756+00:00,2026-05-18T13:52:05.489147+00:00,success,"Pass Hope, Pint by Pint",resp_0f097b997df9f1ad006a0b1318ba208195948738c...,"{'input_tokens': 136, 'input_tokens_details': ...",None,r1__c23b5ca1af5c87dc557b7974,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b128f4a3c8190a292c71ae59c032a
3,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_004,base_004__a1,...,2026-05-18T13:21:24.731889+00:00,2026-05-18T13:52:05.489257+00:00,success,"Pulse Forward, Donate Life",resp_0711b126d6daf80e006a0b136d0d7481958a5029e...,"{'input_tokens': 136, 'input_tokens_details': ...",None,r1__05b8b847be5fbdcb2bcdfcee,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b128f4a3c8190a292c71ae59c032a
4,openai,gpt-5.4,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_005,base_005__a1,...,2026-05-18T13:21:24.732020+00:00,2026-05-18T13:52:05.489333+00:00,success,"Be Someone’s Tomorrow, Donate Blood",resp_00a3904200b23611006a0b136f6a08819690402c5...,"{'input_tokens': 136, 'input_tokens_details': ...",None,r1__503029f248f45d353956e03b,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b128f4a3c8190a292c71ae59c032a


## Cell 14 — Validate Round 1 completeness

This cell checks that the Round 1 batch returned the expected number of successful records.

In [16]:
expected_round1 = len(TASK_SETTINGS) * len(STRATEGIES) * len(agents_df)
actual_round1 = len(round1_df)

print("Expected Round 1 rows:", expected_round1)
print("Actual Round 1 rows:  ", actual_round1)

display(round1_df.groupby(["task_id", "strategy", "condition", "status"]).size().reset_index(name="n"))

if actual_round1 != expected_round1:
    print("WARNING: Row count mismatch. Inspect errors before proceeding.")

if (round1_df["status"] != "success").any():
    print("WARNING: Some Round 1 calls failed or returned empty text. Inspect before proceeding to Round 2.")
    display(round1_df[round1_df["status"] != "success"].head(20))
else:
    print("Round 1 looks complete.")

Expected Round 1 rows: 5400
Actual Round 1 rows:   5400


,task_id,strategy,condition,status,n
0,aut_automobile_tire,diverge,base,success,150
1,aut_automobile_tire,diverge,dyad,success,150
2,aut_automobile_tire,diverge,triad,success,150
3,aut_automobile_tire,vanilla,base,success,150
4,aut_automobile_tire,vanilla,dyad,success,150
5,aut_automobile_tire,vanilla,triad,success,150
6,aut_key,diverge,base,success,150
7,aut_key,diverge,dyad,success,150
8,aut_key,diverge,triad,success,150
9,aut_key,vanilla,base,success,150


Round 1 looks complete.


## Cell 15 — Build the Round 2 request plan

This cell builds Round 2 requests using the saved Round 1 outputs.

For each agent, the prompt includes the agent's own Round 1 output and, depending on condition, zero, one, or two peer outputs from the same Round 1 group.

In [17]:
def get_task_by_id(task_id: str) -> dict:
    for t in TASK_SETTINGS:
        if t["task_id"] == task_id:
            return t
    raise KeyError(task_id)


def build_round2_plan(round1_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    r1_success = round1_df[round1_df["status"] == "success"].copy()

    required_cols = ["task_id", "strategy", "condition", "group_id", "agent_id"]
    if r1_success.duplicated(required_cols).any():
        dupes = r1_success[r1_success.duplicated(required_cols, keep=False)].sort_values(required_cols)
        raise ValueError(f"Duplicate Round 1 successful records:\n{dupes[required_cols + ['text']].head()}")

    for (task_id, strategy, condition, group_id), group in r1_success.groupby(
        ["task_id", "strategy", "condition", "group_id"],
        sort=True,
    ):
        task = get_task_by_id(task_id)
        group = group.sort_values("agent_index").copy()

        expected_group_size = {"base": 1, "dyad": 2, "triad": 3}[condition]
        if len(group) != expected_group_size:
            raise ValueError(
                f"Group size mismatch for {(task_id, strategy, condition, group_id)}: "
                f"expected {expected_group_size}, got {len(group)}"
            )

        for _, ego in group.iterrows():
            peer_rows = group[group["agent_id"] != ego["agent_id"]].sort_values("agent_index")
            peer_texts = peer_rows["text"].tolist()
            peer_agent_ids = peer_rows["agent_id"].tolist()

            user_prompt = build_round2_prompt(
                task=task,
                strategy=strategy,
                condition=condition,
                self_round1=ego["text"],
                peer_round1_texts=peer_texts,
            )

            request_basis = {
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "task_set_id": TASK_SET_ID,
                "round": 2,
                "task_id": task_id,
                "task_family": task["task_family"],
                "strategy": strategy,
                "condition": condition,
                "group_id": group_id,
                "agent_id": ego["agent_id"],
                "agent_index": int(ego["agent_index"]),
                "self_round1_request_key": ego["request_key"],
                "peer_round1_agent_ids": "|".join(peer_agent_ids),
            }
            request_key = "r2__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

            rows.append({
                **request_basis,
                "request_key": request_key,
                "system_instructions": SYSTEM_INSTRUCTIONS,
                "user_prompt": user_prompt,
                "temperature": TEMPERATURE,
                "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                "self_round1_text": ego["text"],
                "peer_round1_texts_json": json.dumps(peer_texts, ensure_ascii=False),
                "created_at_utc": now_iso(),
            })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head()}")

    return plan_df


round2_plan_df = build_round2_plan(round1_df)

expected_round2 = expected_round1

print("Expected Round 2 requests:", expected_round2)
print("Actual Round 2 requests:  ", len(round2_plan_df))

display(round2_plan_df.groupby(["task_id", "strategy", "condition"]).size().reset_index(name="n"))
round2_plan_df.head()

Expected Round 2 requests: 5400
Actual Round 2 requests:   5400


,task_id,strategy,condition,n
0,aut_automobile_tire,diverge,base,150
1,aut_automobile_tire,diverge,dyad,150
2,aut_automobile_tire,diverge,triad,150
3,aut_automobile_tire,vanilla,base,150
4,aut_automobile_tire,vanilla,dyad,150
5,aut_automobile_tire,vanilla,triad,150
6,aut_key,diverge,base,150
7,aut_key,diverge,dyad,150
8,aut_key,diverge,triad,150
9,aut_key,vanilla,base,150


,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,self_round1_request_key,peer_round1_agent_ids,request_key,system_instructions,user_prompt,temperature,max_output_tokens,self_round1_text,peer_round1_texts_json,created_at_utc
0,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_001,base_001__a1,...,r1__fd693e09e906af2d197b15b4,,r2__0cba67d0e83f69365c2ece23,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,A tire lined with acoustic foam can be hung as...,[],2026-05-18T13:52:15.253039+00:00
1,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_002,base_002__a1,...,r1__8aef9f5718f1430f08572fff,,r2__2dbfb9b8e00a835cba77ddf2,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,A tire’s steel-belted rubber tread can be cut ...,[],2026-05-18T13:52:15.253479+00:00
2,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_003,base_003__a1,...,r1__f0038cf824e5686849513c9c,,r2__753fb06500c97acbe5f13a1f,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Cut the tread into interlocking strips to crea...,[],2026-05-18T13:52:15.253824+00:00
3,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_004,base_004__a1,...,r1__d1e346d911ca054b988c7d51,,r2__5ceb328908b112d8f13d0dca,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,A buried automobile tire can serve as a durabl...,[],2026-05-18T13:52:15.254149+00:00
4,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_005,base_005__a1,...,r1__088961c806a7b984425b2110,,r2__5f751df1deb8a3150f96b583,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,A tread-ring section of an automobile tire can...,[],2026-05-18T13:52:15.254462+00:00


## Cell 16 — Write the Round 2 Batch JSONL file

This cell writes the Round 2 batch request file using the same OpenAI Responses API format as Round 1.

In [18]:
round2_jsonl_path, round2_plan_path = make_openai_responses_batch_jsonl(
    plan_df=round2_plan_df,
    round_name="round2",
    output_dir=DIRS["round2_batch_inputs"],
)

round2_jsonl_path, round2_plan_path

Wrote plan:  ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/plans/taskset_b_additional_prompts__round2__openai__gpt-5.4__20260518_095220__plan.csv
Wrote batch: ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/batch_inputs/taskset_b_additional_prompts__round2__openai__gpt-5.4__20260518_095220__batch_input.jsonl
Requests:    5,400


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/batch_inputs/taskset_b_additional_prompts__round2__openai__gpt-5.4__20260518_095220__batch_input.jsonl'),
 PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/plans/taskset_b_additional_prompts__round2__openai__gpt-5.4__20260518_095220__plan.csv'))

## Cell 17 — Submit Round 2 batch

This cell submits the Round 2 batch to OpenAI and saves the local manifest.

In [19]:
round2_batch_info = submit_openai_batch(
    batch_jsonl_path=round2_jsonl_path,
    round_name="round2",
    plan_path=round2_plan_path,
    manifest_dir=DIRS["round2_manifests"],
)

round2_batch_info

Submitted batch:
{
  "run_id": "20260518_091520__368abfc3",
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round2",
  "provider": "openai",
  "model": "gpt-5.4",
  "batch_id": "batch_6a0b1999c6748190824f54a0be99270f",
  "input_file_id": "file-1NHsTDZnDMm1JGXJhuE3Ns",
  "status_at_submission": "validating",
  "submitted_at_utc": "2026-05-18T13:52:25.913798+00:00",
  "batch_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/batch_inputs/taskset_b_additional_prompts__round2__openai__gpt-5.4__20260518_095220__batch_input.jsonl",
  "plan_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/plans/taskset_b_additional_prompts__round2__openai__gpt-5.4__20260518_095220__plan.csv",
  "data_root": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3",
  "manifest_path": "ai_data/

{'run_id': '20260518_091520__368abfc3',
 'task_set_id': 'taskset_b_additional_prompts',
 'round': 'round2',
 'provider': 'openai',
 'model': 'gpt-5.4',
 'batch_id': 'batch_6a0b1999c6748190824f54a0be99270f',
 'input_file_id': 'file-1NHsTDZnDMm1JGXJhuE3Ns',
 'status_at_submission': 'validating',
 'submitted_at_utc': '2026-05-18T13:52:25.913798+00:00',
 'batch_jsonl_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/batch_inputs/taskset_b_additional_prompts__round2__openai__gpt-5.4__20260518_095220__batch_input.jsonl',
 'plan_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/plans/taskset_b_additional_prompts__round2__openai__gpt-5.4__20260518_095220__plan.csv',
 'data_root': 'ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3',
 'manifest_path': 'ai_data/deflect_creativity/openai/model_

## Cell 18 — Check Round 2 batch status

Run this cell periodically until the Round 2 batch status is `completed`.

In [35]:
round2_status = check_openai_batch(round2_batch_info["batch_id"])
round2_status

{
  "batch_id": "batch_6a0b1999c6748190824f54a0be99270f",
  "status": "completed",
  "created_at": 1779112345,
  "in_progress_at": 1779112350,
  "finalizing_at": 1779114232,
  "completed_at": 1779115038,
  "failed_at": null,
  "expired_at": null,
  "cancelled_at": null,
  "request_counts": {
    "total": 5400,
    "completed": 5400,
    "failed": 0
  },
  "output_file_id": "file-1FNBmDmEzEAVD2WNc2iRrc",
  "error_file_id": null,
  "usage": {
    "input_tokens": 2135613,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 587649,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 2723262
  }
}


{'batch_id': 'batch_6a0b1999c6748190824f54a0be99270f',
 'status': 'completed',
 'created_at': 1779112345,
 'in_progress_at': 1779112350,
 'finalizing_at': 1779114232,
 'completed_at': 1779115038,
 'failed_at': None,
 'expired_at': None,
 'cancelled_at': None,
 'request_counts': {'total': 5400, 'completed': 5400, 'failed': 0},
 'output_file_id': 'file-1FNBmDmEzEAVD2WNc2iRrc',
 'error_file_id': None,
 'usage': {'input_tokens': 2135613,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 587649,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 2723262}}

## Cell 19 — Optional reload after kernel restart

Use this cell if the notebook kernel is restarted before the Round 2 batch finishes.

In [ ]:
# Optional reload after kernel restart:
# manifest_path = Path("ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_.../02_round2/manifests/...")
# round2_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(round2_batch_info["data_root"])
# round2_batch_info

## Cell 20 — Download Round 2 raw results

Once the Round 2 batch has completed, this cell downloads the raw JSONL results and any error file.

In [36]:
round2_output_path, round2_error_path = download_openai_batch_results(
    batch_id=round2_batch_info["batch_id"],
    raw_output_dir=DIRS["round2_raw_outputs"],
    raw_error_dir=DIRS["round2_raw_errors"],
    round_name="round2",
)

round2_output_path, round2_error_path

Downloaded output: ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/raw_outputs/taskset_b_additional_prompts__round2__batch_6a0b1999c6748190824f54a0be99270f__output.jsonl
No error file.


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/raw_outputs/taskset_b_additional_prompts__round2__batch_6a0b1999c6748190824f54a0be99270f__output.jsonl'),
 None)

## Cell 21 — Parse Round 2 results

This cell parses the Round 2 batch output into standardized files.

In [37]:
round2_parse_summary = parse_openai_batch_output_to_standard_files(
    batch_output_path=round2_output_path,
    plan_path=Path(round2_batch_info["plan_path"]),
    parsed_dir=DIRS["round2_parsed"],
    round_name="round2",
    batch_id=round2_batch_info["batch_id"],
)

round2_df = pd.read_pickle(round2_parse_summary["parsed_pkl_path"])
print(round2_df.shape)
display(round2_df["status"].value_counts(dropna=False))
round2_df.head()

{
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round2",
  "batch_id": "batch_6a0b1999c6748190824f54a0be99270f",
  "n_records": 5400,
  "n_success": 5400,
  "n_empty_text": 0,
  "n_error": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/parsed/taskset_b_additional_prompts__round2__openai__gpt-5.4__batch_6a0b1999c6748190824f54a0be99270f__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/parsed/taskset_b_additional_prompts__round2__openai__gpt-5.4__batch_6a0b1999c6748190824f54a0be99270f__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/02_round2/parsed/taskset_b_additional_prompts__round2__openai__gpt-5.4__batch_6a0b1999c6748190824f54a0be99270f__parsed.pkl"
}
(5400, 30)


status
success    5400
Name: count, dtype: int64

,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,created_at_utc,parsed_at_utc,status,text,provider_response_id,usage,error,batch_custom_id,batch_output_file,batch_id
0,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_001,base_001__a1,...,2026-05-18T13:52:15.253039+00:00,2026-05-18T14:41:26.450984+00:00,success,A tire mounted on a lazy Susan bearing can ser...,resp_02b5e6afbd475a50006a0b1a396c688190b10a9a0...,"{'input_tokens': 226, 'input_tokens_details': ...",None,r2__0cba67d0e83f69365c2ece23,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b1999c6748190824f54a0be99270f
1,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_002,base_002__a1,...,2026-05-18T13:52:15.253479+00:00,2026-05-18T14:41:26.451417+00:00,success,An automobile tire can be half-buried and pack...,resp_07db45de1f797244006a0b1a39bcf08195bf3bbf1...,"{'input_tokens': 226, 'input_tokens_details': ...",None,r2__2dbfb9b8e00a835cba77ddf2,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b1999c6748190824f54a0be99270f
2,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_003,base_003__a1,...,2026-05-18T13:52:15.253824+00:00,2026-05-18T14:41:26.451898+00:00,success,An old automobile tire can be half-buried upri...,resp_07e228649bfc9548006a0b1a3b7f708197a8a9177...,"{'input_tokens': 223, 'input_tokens_details': ...",None,r2__753fb06500c97acbe5f13a1f,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b1999c6748190824f54a0be99270f
3,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_004,base_004__a1,...,2026-05-18T13:52:15.254149+00:00,2026-05-18T14:41:26.452025+00:00,success,A sliced automobile tire can be bolted around ...,resp_042a81ac769f8baf006a0b1a3e9aa88193b4484d6...,"{'input_tokens': 222, 'input_tokens_details': ...",None,r2__5ceb328908b112d8f13d0dca,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b1999c6748190824f54a0be99270f
4,openai,gpt-5.4,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_005,base_005__a1,...,2026-05-18T13:52:15.254462+00:00,2026-05-18T14:41:26.452143+00:00,success,An automobile tire can be half-buried upright ...,resp_069dfaa3fa39af28006a0b1a3ebedc819081c0fbf...,"{'input_tokens': 225, 'input_tokens_details': ...",None,r2__5f751df1deb8a3150f96b583,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0b1999c6748190824f54a0be99270f


## Cell 22 — Validate Round 2 completeness

This cell checks that the Round 2 batch returned the expected number of successful records.

In [38]:
expected_round2 = expected_round1
actual_round2 = len(round2_df)

print("Expected Round 2 rows:", expected_round2)
print("Actual Round 2 rows:  ", actual_round2)

display(round2_df.groupby(["task_id", "strategy", "condition", "status"]).size().reset_index(name="n"))

if actual_round2 != expected_round2:
    print("WARNING: Row count mismatch. Inspect errors before compiling.")

if (round2_df["status"] != "success").any():
    print("WARNING: Some Round 2 calls failed or returned empty text.")
    display(round2_df[round2_df["status"] != "success"].head(20))
else:
    print("Round 2 looks complete.")

Expected Round 2 rows: 5400
Actual Round 2 rows:   5400


,task_id,strategy,condition,status,n
0,aut_automobile_tire,diverge,base,success,150
1,aut_automobile_tire,diverge,dyad,success,150
2,aut_automobile_tire,diverge,triad,success,150
3,aut_automobile_tire,vanilla,base,success,150
4,aut_automobile_tire,vanilla,dyad,success,150
5,aut_automobile_tire,vanilla,triad,success,150
6,aut_key,diverge,base,success,150
7,aut_key,diverge,dyad,success,150
8,aut_key,diverge,triad,success,150
9,aut_key,vanilla,base,success,150


Round 2 looks complete.


## Cell 23 — Compile analysis-ready long dataset

This cell combines Round 1 and Round 2 records into a single long-format dataset.

The long-format file contains one row per generated output and is suitable for population-level diversity analysis.

In [39]:
def normalize_for_analysis(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in [
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]:
        if col not in out.columns:
            out[col] = None

    keep_cols = [
        "provider",
        "model",
        "task_set_id",
        "round",
        "task_id",
        "task_family",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "agent_index",
        "request_key",
        "status",
        "text",
        "temperature",
        "max_output_tokens",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
        "provider_response_id",
        "usage",
        "error",
        "batch_id",
        "batch_custom_id",
        "batch_output_file",
        "parsed_at_utc",
    ]

    existing_keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[existing_keep_cols].copy()

    out["text_clean"] = out["text"].map(clean_model_text)
    out["is_success"] = out["status"].eq("success")

    return out


round1_analysis_df = normalize_for_analysis(round1_df)
round2_analysis_df = normalize_for_analysis(round2_df)

full_long_df = pd.concat([round1_analysis_df, round2_analysis_df], ignore_index=True)

sort_cols = ["task_id", "strategy", "condition", "group_id", "agent_index", "round"]
full_long_df = full_long_df.sort_values(sort_cols).reset_index(drop=True)

print(full_long_df.shape)
display(full_long_df.groupby(["round", "task_id", "strategy", "condition", "status"]).size().reset_index(name="n").head(30))

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
full_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.csv"
full_pkl_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.pkl"

if full_csv_path.exists() or full_pkl_path.exists():
    raise FileExistsError("Refusing to overwrite compiled long files.")

full_long_df.to_csv(full_csv_path, index=False)
full_long_df.to_pickle(full_pkl_path)

print(full_csv_path)
print(full_pkl_path)

(10800, 29)


,round,task_id,strategy,condition,status,n
0,1,aut_automobile_tire,diverge,base,success,150
1,1,aut_automobile_tire,diverge,dyad,success,150
2,1,aut_automobile_tire,diverge,triad,success,150
3,1,aut_automobile_tire,vanilla,base,success,150
4,1,aut_automobile_tire,vanilla,dyad,success,150
5,1,aut_automobile_tire,vanilla,triad,success,150
6,1,aut_key,diverge,base,success,150
7,1,aut_key,diverge,dyad,success,150
8,1,aut_key,diverge,triad,success,150
9,1,aut_key,vanilla,base,success,150


ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/03_compiled/taskset_b_additional_prompts__openai__gpt-5.4__deflect_creativity__full_long__20260518_104133.csv
ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/03_compiled/taskset_b_additional_prompts__openai__gpt-5.4__deflect_creativity__full_long__20260518_104133.pkl


## Cell 24 — Compile ego-level wide dataset

This cell creates one row per agent, linking each agent's Round 1 and Round 2 outputs.

The wide-format file is useful for self-revision, peer-deflection, and ego-level mechanism analyses.

In [40]:
r1_small = full_long_df[full_long_df["round"].eq(1)].copy()
r2_small = full_long_df[full_long_df["round"].eq(2)].copy()

merge_keys = [
    "provider",
    "model",
    "task_set_id",
    "task_id",
    "task_family",
    "strategy",
    "condition",
    "group_id",
    "agent_id",
    "agent_index",
]

wide_df = r1_small[merge_keys + ["request_key", "status", "text_clean", "batch_id", "usage"]].rename(
    columns={
        "request_key": "round1_request_key",
        "status": "round1_status",
        "text_clean": "round1_text",
        "batch_id": "round1_batch_id",
        "usage": "round1_usage",
    }
).merge(
    r2_small[merge_keys + [
        "request_key",
        "status",
        "text_clean",
        "batch_id",
        "usage",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]].rename(
        columns={
            "request_key": "round2_request_key",
            "status": "round2_status",
            "text_clean": "round2_text",
            "batch_id": "round2_batch_id",
            "usage": "round2_usage",
        }
    ),
    on=merge_keys,
    how="outer",
    validate="one_to_one",
)

wide_df = wide_df.sort_values(["task_id", "strategy", "condition", "group_id", "agent_index"]).reset_index(drop=True)

wide_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.csv"
wide_pkl_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.pkl"

if wide_csv_path.exists() or wide_pkl_path.exists():
    raise FileExistsError("Refusing to overwrite compiled wide files.")

wide_df.to_csv(wide_csv_path, index=False)
wide_df.to_pickle(wide_pkl_path)

print(wide_df.shape)
print(wide_csv_path)
print(wide_pkl_path)
wide_df.head()

(5400, 24)
ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/03_compiled/taskset_b_additional_prompts__openai__gpt-5.4__deflect_creativity__ego_wide__20260518_104133.csv
ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/03_compiled/taskset_b_additional_prompts__openai__gpt-5.4__deflect_creativity__ego_wide__20260518_104133.pkl


,provider,model,task_set_id,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,...,round1_usage,round2_request_key,round2_status,round2_text,round2_batch_id,round2_usage,self_round1_request_key,peer_round1_agent_ids,self_round1_text,peer_round1_texts_json
0,openai,gpt-5.4,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_001,base_001__a1,1,...,"{'input_tokens': 166, 'input_tokens_details': ...",r2__0cba67d0e83f69365c2ece23,success,A tire mounted on a lazy Susan bearing can ser...,batch_6a0b1999c6748190824f54a0be99270f,"{'input_tokens': 226, 'input_tokens_details': ...",r1__fd693e09e906af2d197b15b4,NaN,A tire lined with acoustic foam can be hung as...,[]
1,openai,gpt-5.4,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_002,base_002__a1,1,...,"{'input_tokens': 166, 'input_tokens_details': ...",r2__2dbfb9b8e00a835cba77ddf2,success,An automobile tire can be half-buried and pack...,batch_6a0b1999c6748190824f54a0be99270f,"{'input_tokens': 226, 'input_tokens_details': ...",r1__8aef9f5718f1430f08572fff,NaN,A tire’s steel-belted rubber tread can be cut ...,[]
2,openai,gpt-5.4,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_003,base_003__a1,1,...,"{'input_tokens': 166, 'input_tokens_details': ...",r2__753fb06500c97acbe5f13a1f,success,An old automobile tire can be half-buried upri...,batch_6a0b1999c6748190824f54a0be99270f,"{'input_tokens': 223, 'input_tokens_details': ...",r1__f0038cf824e5686849513c9c,NaN,Cut the tread into interlocking strips to crea...,[]
3,openai,gpt-5.4,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_004,base_004__a1,1,...,"{'input_tokens': 166, 'input_tokens_details': ...",r2__5ceb328908b112d8f13d0dca,success,A sliced automobile tire can be bolted around ...,batch_6a0b1999c6748190824f54a0be99270f,"{'input_tokens': 222, 'input_tokens_details': ...",r1__d1e346d911ca054b988c7d51,NaN,A buried automobile tire can serve as a durabl...,[]
4,openai,gpt-5.4,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_005,base_005__a1,1,...,"{'input_tokens': 166, 'input_tokens_details': ...",r2__5f751df1deb8a3150f96b583,success,An automobile tire can be half-buried upright ...,batch_6a0b1999c6748190824f54a0be99270f,"{'input_tokens': 225, 'input_tokens_details': ...",r1__088961c806a7b984425b2110,NaN,A tread-ring section of an automobile tire can...,[]


## Cell 25 — Basic validation flags

This cell creates simple task-constraint diagnostics.

For slogans, it flags outputs above six words. For stories, it applies a rough sentence counter and flags outputs that do not appear to have exactly eight sentences. These flags are diagnostic and can be used later in robustness analyses.

In [41]:
def word_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b[\w'-]+\b", text))


def sentence_count_rough(text: str) -> int:
    if not isinstance(text, str):
        return 0
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)


validation_df = full_long_df.copy()
validation_df["word_count"] = validation_df["text_clean"].map(word_count)
validation_df["rough_sentence_count"] = validation_df["text_clean"].map(sentence_count_rough)

slogan_violations = validation_df[
    validation_df["task_family"].eq("slogan")
    & validation_df["is_success"]
    & validation_df["word_count"].gt(6)
].copy()

story_sentence_violations = validation_df[
    validation_df["task_family"].eq("story")
    & validation_df["is_success"]
    & validation_df["rough_sentence_count"].ne(8)
].copy()

print("Slogan >6-word violations:", len(slogan_violations))
display(slogan_violations[["round", "task_id", "strategy", "condition", "agent_id", "text_clean", "word_count"]].head(20))

print("Story rough sentence-count violations:", len(story_sentence_violations))
display(story_sentence_violations[["round", "task_id", "strategy", "condition", "agent_id", "text_clean", "rough_sentence_count"]].head(20))

validation_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__validation_flags__{timestamp}.csv"

if validation_csv_path.exists():
    raise FileExistsError(f"Refusing to overwrite validation file: {validation_csv_path}")

validation_df.to_csv(validation_csv_path, index=False)
validation_csv_path

Slogan >6-word violations: 10


,round,task_id,strategy,condition,agent_id,text_clean,word_count
5702,1,slogan_blood_donation,diverge,dyad,dyad_001__a2,Be Someone’s Rare Type of Hero,7
5830,1,slogan_blood_donation,diverge,dyad,dyad_033__a2,Be the Beat in Someone’s Chest,7
5896,1,slogan_blood_donation,diverge,dyad,dyad_050__a1,Be the Beat in Someone’s Veins,7
5982,1,slogan_blood_donation,diverge,dyad,dyad_071__a2,Let Your Pulse Become Someone’s Tomorrow,7
6108,1,slogan_blood_donation,diverge,triad,triad_019__a1,Be the Beat in Someone’s Tomorrow,7
6487,2,slogan_blood_donation,vanilla,base,base_094__a1,"Donate hope, one heartbeat at a time",7
6558,1,slogan_blood_donation,vanilla,base,base_130__a1,Be Someone’s Type When It Matters,7
6749,2,slogan_blood_donation,vanilla,dyad,dyad_038__a1,"Share life, one vein at a time",7
6843,2,slogan_blood_donation,vanilla,dyad,dyad_061__a2,"Donate courage, one pint at a time",7
6875,2,slogan_blood_donation,vanilla,dyad,dyad_069__a2,"Donate Dawn, One Pint at a Time",7


Story rough sentence-count violations: 1355


,round,task_id,strategy,condition,agent_id,text_clean,rough_sentence_count
7200,1,story_horror,diverge,base,base_001__a1,"When the town siren wailed at 2:17 a.m., every...",5
7201,2,story_horror,diverge,base,base_001__a1,"Every night at 1:13, the old elevator in our a...",9
7202,1,story_horror,diverge,base,base_002__a1,"At 2:13 every morning, the baby monitor on Mar...",7
7204,1,story_horror,diverge,base,base_003__a1,"At 2:13 every morning, the old elevator in my ...",7
7205,2,story_horror,diverge,base,base_003__a1,"The summer I worked at the lake camp, Cabin Ni...",7
7206,1,story_horror,diverge,base,base_004__a1,"At 2:17 every morning, the old baby monitor on...",6
7208,1,story_horror,diverge,base,base_005__a1,"Every night at 2:17, the old elevator in my ap...",10
7210,1,story_horror,diverge,base,base_006__a1,"At 2:13 every morning, the old baby monitor on...",6
7212,1,story_horror,diverge,base,base_007__a1,"Every night at 2:13, the old baby monitor in M...",6
7214,1,story_horror,diverge,base,base_008__a1,"Every night at 2:17, the old elevator in our a...",7


PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/03_compiled/taskset_b_additional_prompts__openai__gpt-5.4__deflect_creativity__validation_flags__20260518_104133.csv')

## Cell 26 — Save final manifest

This cell writes a final manifest that records the Round 1 and Round 2 batch IDs, parse summaries, compiled output files, and validation file.

In [42]:
final_manifest = {
    "run_id": RUN_ID,
    "task_set_id": TASK_SET_ID,
    "data_root": str(DATA_ROOT),
    "provider": PROVIDER,
    "model": MODEL_NAME,
    "round1_batch_info": round1_batch_info,
    "round2_batch_info": round2_batch_info,
    "round1_parse_summary": round1_parse_summary,
    "round2_parse_summary": round2_parse_summary,
    "compiled_long_csv": str(full_csv_path),
    "compiled_long_pkl": str(full_pkl_path),
    "compiled_wide_csv": str(wide_csv_path),
    "compiled_wide_pkl": str(wide_pkl_path),
    "validation_csv": str(validation_csv_path),
    "completed_at_utc": now_iso(),
}

final_manifest_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__final_manifest__{timestamp}.json"

if final_manifest_path.exists():
    raise FileExistsError(f"Refusing to overwrite final manifest: {final_manifest_path}")

write_json(final_manifest_path, final_manifest)

final_manifest_path

PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/taskset_b_additional_prompts/run_20260518_091520__368abfc3/03_compiled/taskset_b_additional_prompts__openai__gpt-5.4__deflect_creativity__final_manifest__20260518_104133.json')